# Partie 4 — Entraînement & optimisation des modèles

**Projet :** prédiction du churn client (Telco Customer Churn) — groupe de 5
**Responsable :** Membre 4
**Livrable attendu :** modèles entraînés + tableau comparatif des expériences

## Objectif de cette partie

Les parties 1 à 3 ont fourni un dataset compris, exploré et préparé. Ici, on **entraîne plusieurs modèles de classification**, on les compare de façon **honnête** (sans fuite de données) et on **optimise** les meilleurs. L'évaluation finale détaillée (matrices de confusion, interprétation) est le travail de la partie 5.

## Ce qu'on retient des parties précédentes

| Constat | Conséquence pour la modélisation |
|---|---|
| 26,5 % de churn seulement (classes déséquilibrées) | l'accuracy est trompeuse : un modèle « toujours No » obtient 73,5 %. On suit surtout **Recall, F1 et ROC-AUC** de la classe churn |
| Contrat, ancienneté, fibre, chèque électronique = facteurs de risque | on pourra vérifier que les modèles retrouvent ces variables |
| `tenure`, `TotalCharges`, `MonthlyCharges` très corrélées | attention aux modèles linéaires (régularisation) |
| Découpage train/test déjà fait (80/20, stratifié, `random_state=42`) | on **réutilise exactement le même découpage** ; le test n'est touché qu'à la toute fin |

## Plan du notebook

1. **Mise en place** : configuration, chargement des données, reconstruction du pipeline de prétraitement
2. Stratégie de validation et modèle de référence
3. Entraînement de plusieurs modèles (Logistic Regression, Decision Tree, Random Forest…)
4. Gestion du déséquilibre des classes
5. Optimisation des hyperparamètres
6. Tableau comparatif des expériences et sauvegarde des modèles

## 1. Mise en place

### 1.1 Imports et configuration

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import imblearn
import joblib

# Graine unique pour toute la partie 4 : mêmes résultats à chaque exécution
SEED = 42

# Dossier lu par la partie 4 (produit par la partie 3) et dossiers de sortie
OUT_DIR = Path("outputs")
MODELS_DIR = Path("../models")
REPORTS_DIR = Path("../reports")

pd.set_option("display.max_columns", 60)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"scikit-learn {sklearn.__version__} | imbalanced-learn {imblearn.__version__}")

Python 3.13.7 | pandas 3.1.1 | numpy 2.5.4
scikit-learn 1.9.1 | imbalanced-learn 0.14.2


**Pourquoi afficher les versions ?** Un modèle sauvegardé avec `joblib` ne se recharge correctement qu'avec la même version de scikit-learn. Les versions sont fixées dans `requirements.txt` ; cette cellule permet de vérifier qu'on travaille avec le même environnement que le reste du groupe.

### 1.2 Chargement des données

On part de `telco_clean_features.csv` : les données **nettoyées avec les nouvelles variables, mais avant encodage et mise à l'échelle** (produit par la partie 3).

**Pourquoi ne pas utiliser directement `train_processed.csv` ?** Dans ces fichiers, l'imputation et la standardisation ont déjà été apprises sur *tout* le train. Si on fait ensuite une validation croisée dessus, les statistiques (moyenne, écart-type) de chaque pli de validation ont déjà « fuité » dans l'entraînement : c'est une fuite légère, mais réelle. En repartant des données avant encodage, on pourra remettre le prétraitement **à l'intérieur** de chaque pli de validation croisée.

In [2]:
df = pd.read_csv(OUT_DIR / "telco_clean_features.csv")
print("Dimensions :", df.shape)
print("Valeurs manquantes par colonne (uniquement celles > 0) :")
print(df.isna().sum()[lambda s: s > 0])
df.head(3)

Dimensions : (7043, 45)
Valeurs manquantes par colonne (uniquement celles > 0) :
Series([], dtype: int64)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group,is_new_customer,tenure_years,has_internet,has_phone,n_security_services,n_streaming_services,n_services_total,has_protection_bundle,internet_no_protection,avg_monthly_spend,charge_delta,charge_ratio,charge_per_service,log_total_charges,is_month_to_month,contract_length_m,auto_payment,is_electronic_check,has_family,is_alone,fiber_month_to_month,new_and_mtm,senior_alone
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,0-6m,1,0.083333,1,0,1,0,2,0,0,29.850000,0.000000,1.000000,14.9250,3.429137,1,1,0,1,1,0,0,1,0
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0,25-48m,0,2.833333,1,1,2,0,4,0,0,55.573529,1.376471,1.024768,14.2375,7.544597,0,12,0,0,0,1,0,0,0
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,0-6m,1,0.166667,1,1,2,0,4,0,0,54.075000,-0.225000,0.995839,13.4625,4.692723,1,1,0,0,0,1,0,1,0


### 1.3 Découpage train / test (identique à la partie 3)

Mêmes paramètres que la partie 3 : 80 % / 20 %, stratifié sur `Churn`, `random_state=42`. Le **jeu de test est mis de côté** : il ne servira qu'à la fin, pour évaluer le modèle retenu. Toutes les décisions (choix du modèle, réglage des hyperparamètres) se feront uniquement avec le train.

In [3]:
from sklearn.model_selection import train_test_split

TARGET = "Churn"
X = df.drop(columns=[TARGET, "customerID"])   # customerID est un identifiant, pas une variable explicative
y = df[TARGET]
ids = df["customerID"]

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.20, stratify=y, random_state=SEED)

print(f"Train : {X_train.shape} | churn = {y_train.mean():.2%}")
print(f"Test  : {X_test.shape}  | churn = {y_test.mean():.2%}")

Train : (5634, 43) | churn = 26.54%
Test  : (1409, 43)  | churn = 26.54%


### 1.4 Vérification : est-ce le même découpage que la partie 3 ?

Si le groupe n'utilise pas les mêmes clients dans le train et le test, les résultats des parties 4 et 5 ne seront pas comparables. On compare donc les identifiants avec les fichiers produits par la partie 3.

In [4]:
ref_train = pd.read_csv(OUT_DIR / "train_processed.csv", usecols=["customerID", "Churn"])
ref_test = pd.read_csv(OUT_DIR / "test_processed.csv", usecols=["customerID", "Churn"])

assert set(id_train) == set(ref_train["customerID"]), "Le train n'est pas celui de la partie 3"
assert set(id_test) == set(ref_test["customerID"]), "Le test n'est pas celui de la partie 3"
assert set(id_train).isdisjoint(set(id_test)), "Des clients sont à la fois dans le train et le test"
print("OK : train et test identiques à ceux de la partie 3 "
      f"({len(id_train)} + {len(id_test)} clients, aucun client en commun).")

OK : train et test identiques à ceux de la partie 3 (5634 + 1409 clients, aucun client en commun).


**Résultat :** on travaille bien sur les mêmes clients que le reste du groupe, avec la même proportion de churn (26,54 %) dans le train et dans le test.

### 1.5 Reconstruction du pipeline de prétraitement

On reprend **exactement les mêmes choix que la partie 3** (mêmes colonnes, mêmes imputations, même encodage), mais sous forme d'une fonction `make_preprocessor()` qui renvoie un pipeline **neuf et non ajusté**.

**Pourquoi une fonction ?** En validation croisée, il faut ré-apprendre le prétraitement sur chaque pli d'entraînement. Chaque pli reçoit donc son propre pipeline vierge, qui n'a jamais vu les données de validation.

Rappel des choix de la partie 3 :
- **Numériques** : imputation par la médiane, puis standardisation (utile aux modèles sensibles aux échelles, comme la régression logistique)
- **Binaires 0/1** : conservées telles quelles
- **`tenure_group`** : encodage ordinal (les tranches d'ancienneté sont ordonnées)
- **Nominales** : one-hot (`drop="if_binary"` garde une seule colonne pour les variables Yes/No)
- Les 6 variables exactement redondantes (`has_phone`, `has_internet`, `is_electronic_check`, `is_month_to_month`, `tenure_years`, `is_alone`) ne sont pas utilisées : elles sont ignorées par `remainder="drop"`.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges", "log_total_charges",
                    "avg_monthly_spend", "charge_delta", "charge_ratio", "charge_per_service",
                    "n_security_services", "n_streaming_services", "n_services_total", "contract_length_m"]

binary_passthrough = ["is_new_customer", "has_protection_bundle", "internet_no_protection",
                      "auto_payment", "has_family", "fiber_month_to_month", "new_and_mtm", "senior_alone"]

ordinal_features = ["tenure_group"]
ordinal_categories = [["0-6m", "7-12m", "13-24m", "25-48m", "49-72m"]]

nominal_features = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines",
                    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
                    "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod"]

redundant_features = ["has_phone", "has_internet", "is_electronic_check",
                      "is_month_to_month", "tenure_years", "is_alone"]


def make_preprocessor():
    """Renvoie un prétraitement NEUF (non ajusté), identique à celui de la partie 3."""
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                             ("scaler", StandardScaler())])
    ordinal_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                             ("encoder", OrdinalEncoder(categories=ordinal_categories))])
    nominal_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                             ("encoder", OneHotEncoder(drop="if_binary", handle_unknown="infrequent_if_exist",
                                                       min_frequency=0.01, sparse_output=False))])
    return ColumnTransformer(
        transformers=[("num", numeric_pipe, numeric_features),
                      ("bin", "passthrough", binary_passthrough),
                      ("ord", ordinal_pipe, ordinal_features),
                      ("nom", nominal_pipe, nominal_features)],
        remainder="drop",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


# Garde-fou : toutes les colonnes de X sont soit utilisées, soit explicitement écartées
used = numeric_features + binary_passthrough + ordinal_features + nominal_features
assert len(used) == len(set(used)), "Colonne utilisée deux fois"
assert set(used) | set(redundant_features) == set(X.columns), set(X.columns) ^ (set(used) | set(redundant_features))
print(f"{len(used)} variables utilisées, {len(redundant_features)} écartées (redondantes).")

37 variables utilisées, 6 écartées (redondantes).


### 1.6 Vérification : le pipeline reconstruit redonne-t-il les fichiers de la partie 3 ?

On ajuste le nouveau pipeline sur le train et on compare le résultat, valeur par valeur, avec `train_processed.csv` et `test_processed.csv` (produits par la partie 3). Si tout est identique, on est sûr de ne pas avoir changé le prétraitement en le reconstruisant.

In [6]:
prep_check = make_preprocessor()
Xtr_p = prep_check.fit_transform(X_train)   # fit sur le train uniquement
Xte_p = prep_check.transform(X_test)

feature_names_p3 = pd.read_csv(OUT_DIR / "feature_names.csv")["feature"].tolist()
assert list(Xtr_p.columns) == feature_names_p3, "Les noms/ordre des colonnes diffèrent de la partie 3"

for nom, mine, fichier, ids_ in [("train", Xtr_p, "train_processed.csv", id_train),
                                 ("test", Xte_p, "test_processed.csv", id_test)]:
    ref = pd.read_csv(OUT_DIR / fichier).set_index("customerID").loc[ids_.values]   # même ordre de clients
    ecart = np.abs(mine.to_numpy() - ref[feature_names_p3].to_numpy()).max()
    assert ecart < 1e-6, f"{nom} : écart max {ecart}"
    print(f"{nom:5s}: {mine.shape} - écart maximal avec la partie 3 = {ecart:.2e}")

print("OK : le pipeline reconstruit est identique à celui de la partie 3.")

train: (5634, 58) - écart maximal avec la partie 3 = 0.00e+00
test : (1409, 58) - écart maximal avec la partie 3 = 0.00e+00
OK : le pipeline reconstruit est identique à celui de la partie 3.


**Résultat :** le pipeline est identique à celui de la partie 3 (mêmes 58 variables, mêmes valeurs). La différence est dans l'usage : à partir de maintenant, il sera appelé *à l'intérieur* de chaque pli de validation croisée, ce qui garantit qu'aucune information du pli de validation n'est utilisée pour apprendre l'imputation ou la standardisation.

**Fin de l'étape 1.** Prochaine étape : définir la stratégie de validation et le modèle de référence.

## 2. Stratégie de validation et modèle de référence

### 2.1 Pourquoi une validation croisée ?

Pour **comparer des modèles et régler leurs hyperparamètres**, il faut pouvoir les juger sur des données qu'ils n'ont pas vues à l'entraînement. Mais on ne peut pas utiliser le jeu de test pour ça : si on choisit le modèle qui obtient le meilleur score sur le test, ce score devient optimiste (le test a influencé notre choix) et ne mesure plus la performance réelle.

La solution est la **validation croisée** *à l'intérieur du train* :
1. on coupe le train en 5 parties (« plis ») ;
2. on entraîne sur 4 plis et on évalue sur le 5ᵉ, puis on recommence 5 fois en changeant le pli de validation ;
3. on fait la **moyenne** des 5 scores (et on regarde leur écart-type pour mesurer la stabilité).

Le jeu de test, lui, reste intact jusqu'à la toute fin.

### 2.2 Découpage en plis

On utilise `StratifiedKFold` :
- **stratifié** : chaque pli garde environ 26,5 % de churn, ce qui est essentiel avec des classes déséquilibrées ;
- `shuffle=True` avec `random_state=SEED` : les clients sont mélangés avant la découpe, de façon reproductible.

In [7]:
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Contrôle : taille et taux de churn de chaque pli de validation
lignes = []
for i, (idx_tr, idx_val) in enumerate(cv.split(X_train, y_train), start=1):
    lignes.append({"pli": i, "n_entrainement": len(idx_tr), "n_validation": len(idx_val),
                   "churn_validation": f"{y_train.iloc[idx_val].mean():.2%}"})
pd.DataFrame(lignes).set_index("pli")

,n_entrainement,n_validation,churn_validation
pli,,,
1,4507,1127,26.53%
2,4507,1127,26.53%
3,4507,1127,26.53%
4,4507,1127,26.53%
5,4508,1126,26.55%


### 2.3 Métriques suivies

Avec 26,5 % de churn, l'**accuracy** seule est trompeuse : un modèle qui répond toujours « le client reste » aurait 73,5 % d'accuracy tout en ne détectant **aucun** client qui part. On suit donc plusieurs métriques, en considérant « churn » comme la classe positive :

| Métrique | Question à laquelle elle répond |
|---|---|
| **Recall** (rappel) | parmi les clients qui partent vraiment, combien en détecte-t-on ? |
| **Precision** | parmi les clients signalés comme à risque, combien partent vraiment ? |
| **F1-score** | compromis entre precision et recall |
| **ROC-AUC** | capacité à classer les clients du plus au moins à risque, indépendamment du seuil de décision |
| Accuracy | à titre indicatif seulement |

**Métrique de décision : le F1-score de la classe churn**, avec le Recall en seconde lecture. Le document de la partie 1 indique que rater un client qui part coûte plus cher que lancer une action de fidélisation inutile : le recall compte donc beaucoup, mais le F1 évite de gagner du recall en signalant tout le monde.

In [8]:
from sklearn.metrics import make_scorer, precision_score, recall_score

# zero_division=0 : si un modèle ne prédit jamais « churn », precision = 0 (au lieu d'un avertissement)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": "f1",
    "roc_auc": "roc_auc",
}
print("Métriques calculées à chaque pli :", list(scoring))

Métriques calculées à chaque pli : ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']


### 2.4 Fonction d'expérience et journal des résultats

Le TP demande de **consigner les paramètres et les résultats de chaque expérience**. On crée :
- une liste `experiences` qui garde une ligne par expérience ;
- une fonction `evaluer(nom, modele, ...)` qui place le modèle derrière le prétraitement dans un `Pipeline`, lance la validation croisée sur le train, et enregistre les moyennes et écarts-types.

Le prétraitement est **dans** le pipeline : il est donc ré-appris sur les 4 plis d'entraînement à chaque tour, sans jamais voir le pli de validation.

In [9]:
import time
from imblearn.pipeline import Pipeline as ImbPipeline   # comme Pipeline, mais accepte un rééchantillonneur (SMOTE)
from sklearn.model_selection import cross_validate

experiences = []   # journal : une ligne par expérience


def evaluer(nom, modele, remarque="", sampler=None):
    """Validation croisée (prétraitement, et éventuel rééchantillonnage, inclus dans chaque pli) + journal.

    Le rééchantillonneur (ex. SMOTE) n'agit qu'à l'entraînement de chaque pli : le pli de validation
    garde sa vraie répartition de classes."""
    etapes = [("prep", make_preprocessor())]
    if sampler is not None:
        etapes.append(("reechantillonnage", sampler))
    etapes.append(("modele", modele))
    pipe = ImbPipeline(etapes)
    debut = time.time()
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    ligne = {"experience": nom}
    for m in scoring:
        ligne[f"{m}_moy"] = res[f"test_{m}"].mean()
        ligne[f"{m}_std"] = res[f"test_{m}"].std()
    ligne["temps_s"] = time.time() - debut
    ligne["parametres"] = str(modele.get_params(deep=False)) + (f" | {sampler}" if sampler is not None else "")
    ligne["remarque"] = remarque
    experiences.append(ligne)
    return ligne


def tableau():
    """Tableau lisible du journal, trié par F1 décroissant."""
    cols = ["accuracy_moy", "precision_moy", "recall_moy", "f1_moy", "f1_std", "roc_auc_moy", "temps_s"]
    return pd.DataFrame(experiences).set_index("experience")[cols].sort_values("f1_moy", ascending=False).round(3)

### 2.5 Modèle de référence (baseline)

Un modèle de référence donne le **score minimal à battre** : si un vrai modèle ne fait pas mieux, il n'apporte rien. On prend le classifieur le plus simple possible, `DummyClassifier(strategy="most_frequent")`, qui répond toujours « le client reste » (la classe majoritaire).

In [10]:
from sklearn.dummy import DummyClassifier

evaluer("Baseline - toujours 'reste'", DummyClassifier(strategy="most_frequent"),
        remarque="classe majoritaire")
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Baseline - toujours 'reste',0.735,0.0,0.0,0.0,0.0,0.5,1.011


**Lecture du résultat :**
- **Accuracy = 73,5 %** : ça ressemble à un bon score, mais c'est uniquement la proportion de clients qui restent. Le modèle ne fait que répéter la classe majoritaire.
- **Recall = 0 et F1 = 0** : il ne détecte aucun des clients qui partent, ce qui est justement le but du projet.
- **ROC-AUC = 0,5** : c'est le niveau du hasard, le modèle ne sait pas classer les clients.

Cela confirme le choix des métriques : l'accuracy seule aurait fait passer ce modèle inutile pour un bon modèle. **Tout vrai modèle devra faire mieux que F1 = 0 et ROC-AUC = 0,5**, et surtout se juger sur le F1 et le recall de la classe churn.

**Fin de l'étape 2.** Prochaine étape : entraîner plusieurs modèles (Logistic Regression, Decision Tree, Random Forest) avec leurs paramètres par défaut.

## 3. Entraînement de plusieurs modèles (paramètres par défaut)

On entraîne d'abord trois modèles classiques **avec leurs réglages par défaut**, pour voir ce que chacun donne « tel quel » avant toute optimisation. Ils sont évalués par la même validation croisée que le modèle de référence, donc directement comparables.

| Modèle | Principe | Points forts | Points faibles |
|---|---|---|---|
| **Régression logistique** | combine les variables avec des poids pour estimer une probabilité de churn | simple, rapide, interprétable (un poids par variable) | ne capte que des effets linéaires ; sensible aux variables corrélées |
| **Arbre de décision** | pose une suite de questions « oui/non » (ex. contrat mensuel ? ancienneté < 6 mois ?) | très lisible, capte les interactions | tend à **sur-apprendre** (il mémorise le train) quand on ne limite pas sa profondeur |
| **Forêt aléatoire** | moyenne de nombreux arbres entraînés sur des échantillons différents | robuste, performante en général, limite le sur-apprentissage | moins lisible, plus lente |

À chaque fois, `random_state=SEED` garantit des résultats reproductibles.

### 3.1 Régression logistique

On fixe `max_iter=1000` (au lieu de 100 par défaut) uniquement pour laisser l'algorithme converger ; les autres paramètres sont ceux par défaut, notamment la régularisation `C=1.0`.

In [11]:
from sklearn.linear_model import LogisticRegression

evaluer("Régression logistique (défaut)", LogisticRegression(max_iter=1000, random_state=SEED))
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,1.212
Baseline - toujours 'reste',0.735,0.000,0.000,0.000,0.000,0.500,1.011


### 3.2 Arbre de décision

Sans aucune limite de profondeur : l'arbre peut grandir jusqu'à classer parfaitement chaque client du train.

In [12]:
from sklearn.tree import DecisionTreeClassifier

evaluer("Arbre de décision (défaut)", DecisionTreeClassifier(random_state=SEED))
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,1.212
Arbre de décision (défaut),0.728,0.489,0.512,0.500,0.014,0.660,1.127
Baseline - toujours 'reste',0.735,0.000,0.000,0.000,0.000,0.500,1.011


### 3.3 Forêt aléatoire

Réglages par défaut : 100 arbres, sans limite de profondeur.

In [13]:
from sklearn.ensemble import RandomForestClassifier

evaluer("Forêt aléatoire (défaut)", RandomForestClassifier(random_state=SEED))
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,1.212
Forêt aléatoire (défaut),0.791,0.638,0.495,0.557,0.026,0.828,4.493
Arbre de décision (défaut),0.728,0.489,0.512,0.500,0.014,0.660,1.127
Baseline - toujours 'reste',0.735,0.000,0.000,0.000,0.000,0.500,1.011


### 3.4 Vérification du sur-apprentissage : score sur le train vs score en validation

Un modèle qui **sur-apprend** a un excellent score sur les données qu'il a vues et un score bien plus faible sur des données nouvelles. On compare donc, pour chaque modèle, le F1 mesuré sur les plis d'entraînement et celui mesuré sur les plis de validation. Un grand écart signale du sur-apprentissage.

In [14]:
modeles_defaut = {
    "Régression logistique": LogisticRegression(max_iter=1000, random_state=SEED),
    "Arbre de décision": DecisionTreeClassifier(random_state=SEED),
    "Forêt aléatoire": RandomForestClassifier(random_state=SEED),
}
lignes = []
for nom, m in modeles_defaut.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("modele", m)])
    r = cross_validate(pipe, X_train, y_train, cv=cv, scoring="f1", return_train_score=True, n_jobs=1)
    lignes.append({"modèle": nom, "F1 train": r["train_score"].mean(), "F1 validation": r["test_score"].mean()})
ecarts = pd.DataFrame(lignes).set_index("modèle")
ecarts["écart"] = ecarts["F1 train"] - ecarts["F1 validation"]
ecarts.round(3)

,F1 train,F1 validation,écart
modèle,,,
Régression logistique,0.603,0.596,0.007
Arbre de décision,0.997,0.500,0.497
Forêt aléatoire,0.997,0.557,0.439


### 3.5 Lecture des résultats

| Modèle (défaut) | F1 | Recall | ROC-AUC | F1 train → validation |
|---|---|---|---|---|
| Régression logistique | **0,596** | **0,536** | **0,849** | 0,603 → 0,596 |
| Forêt aléatoire | 0,557 | 0,495 | 0,828 | 0,997 → 0,557 |
| Arbre de décision | 0,500 | 0,512 | 0,660 | 0,997 → 0,500 |
| Référence (toujours « reste ») | 0 | 0 | 0,500 | — |

**Ce qu'on observe**
1. **Tous les modèles battent la référence**, ce qui montre que les variables contiennent bien de l'information sur le churn.
2. **La régression logistique est la meilleure** sur les trois métriques de décision, et la plus **stable** : son score est quasi identique sur le train et en validation (écart de 0,007). Elle ne sur-apprend pas.
3. **L'arbre et la forêt sur-apprennent fortement** : F1 de 0,997 sur le train (ils retiennent presque par cœur les clients vus) mais 0,500 et 0,557 en validation. Sans limite de profondeur, ils ont mémorisé le bruit du train au lieu d'apprendre des règles générales.
4. L'arbre seul est même **moins bon en accuracy (72,8 %) que le modèle de référence (73,5 %)** : un modèle qui sur-apprend peut faire pire qu'une règle triviale.
5. **Le recall reste modeste (≈ 50 %)** : avec les réglages par défaut, on rate environ un client sur deux qui part. C'est la principale piste d'amélioration.

**Ce qu'on en déduit pour la suite**
- Pour l'arbre et la forêt, le problème n'est pas le modèle mais ses **réglages** : il faut limiter la complexité (profondeur maximale, taille minimale des feuilles). C'est typiquement le rôle de l'optimisation d'hyperparamètres (étape 5).
- Pour améliorer le recall sur tous les modèles, on va tester la **gestion du déséquilibre des classes** (étape 4) : donner plus de poids à la classe « churn » ou rééquilibrer les données.

**Fin de l'étape 3.** Prochaine étape : gérer le déséquilibre des classes.